# Within t+2 모델링 (24시간 이내 발생 예측)

- LR/RF/XGB/LightGBM/MLP baseline은 anchor `t` 시점 feature만 입력으로 사용
- LSTM은 `t-3~t` 4개 time step을 입력으로 사용
- target은 `t~t+2` 중 delirium이 한 번이라도 발생하는지 여부
- binary within target에는 horizon별 mask가 없으므로 full `t~t+2` target이 모두 관측된 row만 사용


In [24]:
from pathlib import Path
import json
import random

import joblib
import lightgbm
import numpy as np
import optuna
import pandas as pd
import torch
import xgboost
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


In [25]:
# 작업 위치
PROJECT_DIR  = Path.cwd().resolve().parent
SRC_DIR      = PROJECT_DIR / "src"
MODELING_DIR = PROJECT_DIR / "processed" / "data_split"
MODEL_DIR    = PROJECT_DIR / "models" / "within_t_plus_2"
OUTPUT_DIR   = PROJECT_DIR / "outputs" / "modeling" / "within_t_plus_2"


In [26]:
# 설정값 (config)
RANDOM_STATE  = 42
N_FOLDS       = 5
N_TRIALS_ML   = 30
N_TRIALS_MLP  = 30
N_TRIALS_LSTM = 30
MAX_EPOCHS    = 25
PATIENCE      = 5

ML_MODEL_NAMES = ["LR", "RF", "XGB", "LGBM"]
RUN_MLP        = True
RUN_LSTM       = True


In [27]:
# CUDA GPU 사용 설정
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("device:", device)
    print("gpu:", torch.cuda.get_device_name(0))
    print("cuda version:", torch.version.cuda)
else:
    device = torch.device("cpu")
    print("device:", device)
    print("CUDA GPU를 찾지 못해 CPU로 실행합니다.")

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.set_num_threads(2)


device: cpu
CUDA GPU를 찾지 못해 CPU로 실행합니다.


## 전처리 산출물 로딩


In [28]:
# 모델링에 필요한 입력 파일 경로
required_files = {
    "X_train"                : MODELING_DIR / "X_train_lstm.npy",
    "X_test"                 : MODELING_DIR / "X_test_lstm.npy",
    "y_train_within_t_plus_2": MODELING_DIR / "y_train_lstm.npy",
    "y_test_within_t_plus_2" : MODELING_DIR / "y_test_lstm.npy",
    "y_train_steps"          : MODELING_DIR / "y_train_steps_lstm.npy",
    "y_test_steps"           : MODELING_DIR / "y_test_steps_lstm.npy",
    "y_train_step_mask"      : MODELING_DIR / "y_train_step_mask_lstm.npy",
    "y_test_step_mask"       : MODELING_DIR / "y_test_step_mask_lstm.npy",
    "meta_train"             : MODELING_DIR / "lstm_train_metadata.csv",
    "meta_test"              : MODELING_DIR / "lstm_test_metadata.csv",
}


In [29]:
# LSTM 입력 tensor와 target array 로딩
X_train                 = np.load(required_files["X_train"]).astype(np.float32)
X_test                  = np.load(required_files["X_test"]).astype(np.float32)
y_train_within_t_plus_2 = np.load(required_files["y_train_within_t_plus_2"]).astype(np.float32)
y_test_within_t_plus_2  = np.load(required_files["y_test_within_t_plus_2"]).astype(np.float32)
y_train_steps           = np.load(required_files["y_train_steps"]).astype(np.float32)
y_test_steps            = np.load(required_files["y_test_steps"]).astype(np.float32)
y_train_step_mask       = np.load(required_files["y_train_step_mask"]).astype(np.float32)
y_test_step_mask        = np.load(required_files["y_test_step_mask"]).astype(np.float32)

# train/test metadata 로딩
meta_train = pd.read_csv(required_files["meta_train"])
meta_test  = pd.read_csv(required_files["meta_test"])

# binary within target은 full t~t+2 window가 모두 관측된 row만 사용
train_keep = meta_train["target_available_count"].to_numpy() >= 3
test_keep  = meta_test["target_available_count"].to_numpy() >= 3

X_train                 = X_train[train_keep]
X_test                  = X_test[test_keep]
y_train_within_t_plus_2 = y_train_within_t_plus_2[train_keep]
y_test_within_t_plus_2  = y_test_within_t_plus_2[test_keep]
y_train_steps           = y_train_steps[train_keep]
y_test_steps            = y_test_steps[test_keep]
y_train_step_mask       = y_train_step_mask[train_keep]
y_test_step_mask        = y_test_step_mask[test_keep]
meta_train              = meta_train.loc[train_keep].reset_index(drop=True)
meta_test               = meta_test.loc[test_keep].reset_index(drop=True)

# LR/RF/XGB/LightGBM/MLP baseline 입력: anchor t 시점 feature
X_train_t = X_train[:, -1, :].astype(np.float32)
X_test_t  = X_test[:, -1, :].astype(np.float32)


In [30]:
# 로딩된 데이터의 기본 형태와 target 분포 확인
data_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "n_samples": X_train.shape[0],
            "sequence_length": X_train.shape[1],
            "n_features": X_train.shape[2],
            "within_t_plus_2_positive_rate": y_train_within_t_plus_2.mean(),
            "nan_count": np.isnan(X_train).sum(),
        },
        {
            "split": "test",
            "n_samples": X_test.shape[0],
            "sequence_length": X_test.shape[1],
            "n_features": X_test.shape[2],
            "within_t_plus_2_positive_rate": y_test_within_t_plus_2.mean(),
            "nan_count": np.isnan(X_test).sum(),
        },
    ]
)

display(data_summary)


,split,n_samples,sequence_length,n_features,within_t_plus_2_positive_rate,nan_count
0,train,3738,4,139,0.394596,0
1,test,1590,4,139,0.390566,0


## Subject-level cross-validation split

같은 환자의 window가 train과 validation fold에 동시에 들어가지 않도록 `subject_id` 기준으로 K-fold CV를 구성

In [31]:
# 환자 단위 stratified K-fold 생성 함수 정의
def make_subject_cv_folds(meta: pd.DataFrame, y_within_t_plus_2: np.ndarray, n_folds: int, random_state: int) -> list[dict]:
    subject_summary = meta.assign(y_within_t_plus_2=y_within_t_plus_2).groupby("subject_id", as_index=False)["y_within_t_plus_2"].max()
    subjects = subject_summary["subject_id"].to_numpy()
    labels = subject_summary["y_within_t_plus_2"].to_numpy(dtype=int)

    splitter = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    split_iter = splitter.split(subjects, labels)

    folds = []
    subject_by_row = meta["subject_id"].to_numpy()
    for fold_id, (train_idx, val_idx) in enumerate(split_iter, start=1):
        train_subjects = set(subjects[train_idx])
        val_subjects = set(subjects[val_idx])
        train_mask = np.isin(subject_by_row, list(train_subjects))
        val_mask = np.isin(subject_by_row, list(val_subjects))
        folds.append(
            {
                "fold_id": fold_id,
                "train_mask": train_mask,
                "val_mask": val_mask,
                "n_train_subjects": len(train_subjects),
                "n_val_subjects": len(val_subjects),
                "train_positive_rate": float(y_within_t_plus_2[train_mask].mean()),
                "val_positive_rate": float(y_within_t_plus_2[val_mask].mean()),
            }
        )

    return folds


In [32]:
# CV fold 생성
cv_folds = make_subject_cv_folds(meta_train, y_train_within_t_plus_2, N_FOLDS, RANDOM_STATE)
print(f"Using {len(cv_folds)} subject-level CV folds")


Using 5 subject-level CV folds


In [33]:
# fold별 train/validation 분리 결과 확인
cv_summary = pd.DataFrame(
    [
        {
            "fold_id": fold["fold_id"],
            "train_sequences": int(fold["train_mask"].sum()),
            "validation_sequences": int(fold["val_mask"].sum()),
            "train_subjects": fold["n_train_subjects"],
            "validation_subjects": fold["n_val_subjects"],
            "train_within_t_plus_2_positive_rate": fold["train_positive_rate"],
            "validation_within_t_plus_2_positive_rate": fold["val_positive_rate"],
        }
        for fold in cv_folds
    ]
)

display(cv_summary)


,fold_id,train_sequences,validation_sequences,train_subjects,validation_subjects,train_within_t_plus_2_positive_rate,validation_within_t_plus_2_positive_rate
0,1,3065,673,268,68,0.407178,0.337296
1,2,2761,977,269,67,0.390076,0.407369
2,3,2965,773,269,67,0.392243,0.403622
3,4,2946,792,269,67,0.379837,0.449495
4,5,3215,523,269,67,0.402177,0.347992


## 모델링 함수 정의


In [34]:
def to_builtin(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {key: to_builtin(val) for key, val in value.items()}
    if isinstance(value, list):
        return [to_builtin(val) for val in value]
    return value


In [35]:
def binary_metric_dict(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    y_true = y_true.astype(int)
    y_pred = (y_prob >= threshold).astype(int)
    if len(y_true) == 0:
        return {
            "auroc": np.nan,
            "auprc": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "ppv": np.nan,
            "npv": np.nan,
            "tp": 0,
            "fp": 0,
            "tn": 0,
            "fn": 0,
        }

    if len(np.unique(y_true)) == 2:
        auroc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
    else:
        auroc = np.nan
        auprc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "auroc": auroc,
        "auprc": auprc,
        "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "ppv": tp / (tp + fp) if (tp + fp) else np.nan,
        "npv": tn / (tn + fn) if (tn + fn) else np.nan,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


def positive_weight(y: np.ndarray) -> float:
    pos = float(y.sum())
    neg = float(len(y) - pos)
    return neg / max(pos, 1.0)


## ML baseline 공통 함수


In [36]:
def fit_ml_estimator(model_name: str, estimator, x_train: np.ndarray, y_train: np.ndarray, use_scaler: bool):
    scaler = StandardScaler() if use_scaler else None
    x_fit = scaler.fit_transform(x_train) if scaler is not None else x_train
    try:
        estimator.fit(x_fit, y_train.astype(int))
        return estimator, scaler, None
    except Exception as exc:
        if model_name == "XGB" and (
            estimator.get_params().get("device") == "cuda"
            or estimator.get_params().get("tree_method") == "gpu_hist"
        ):
            estimator = clone(estimator)
            cpu_params = {"tree_method": "hist"}
            if "device" in estimator.get_params():
                cpu_params["device"] = "cpu"
            if "predictor" in estimator.get_params():
                cpu_params["predictor"] = "auto"
            estimator.set_params(**cpu_params)
            estimator.fit(x_fit, y_train.astype(int))
            return estimator, scaler, f"XGBoost GPU fit failed and retrained on CPU: {exc}"
        if model_name == "LGBM" and estimator.get_params().get("device_type") == "gpu":
            estimator = clone(estimator)
            estimator.set_params(device_type="cpu")
            estimator.fit(x_fit, y_train.astype(int))
            return estimator, scaler, f"LightGBM GPU fit failed and retrained on CPU: {exc}"
        raise


def predict_ml_proba(estimator, scaler, x: np.ndarray) -> np.ndarray:
    x_eval = scaler.transform(x) if scaler is not None else x
    return estimator.predict_proba(x_eval)[:, 1]


## Logistic Regression baseline


In [37]:
def suggest_lr_params(trial: optuna.Trial) -> dict:
    return {
        "C": trial.suggest_float("C", 1e-3, 1e2, log=True),
        "penalty": trial.suggest_categorical("penalty", ["l1", "l2"]),
    }


def build_lr_estimator(params: dict, random_state: int):
    return LogisticRegression(
        **params,
        solver="liblinear",
        class_weight="balanced",
        max_iter=2000,
        random_state=random_state,
    )


## Random Forest baseline


In [38]:
def suggest_rf_params(trial: optuna.Trial) -> dict:
    return {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
        "max_depth": trial.suggest_categorical("max_depth", [None, 4, 8, 12, 16, 24]),
        "min_samples_split": trial.suggest_categorical("min_samples_split", [2, 5, 10, 20]),
        "min_samples_leaf": trial.suggest_categorical("min_samples_leaf", [1, 2, 4, 8]),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5]),
    }


def build_rf_estimator(params: dict, random_state: int):
    return RandomForestClassifier(
        **params,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=random_state,
    )

## XGBoost baseline


In [ ]:
def xgb_gpu_params(use_gpu: bool) -> dict:
    if not use_gpu:
        return {"tree_method": "hist"}

    major = int(xgboost.__version__.split(".")[0])
    if major >= 2:
        return {"tree_method": "hist", "device": "cuda"}
    return {"tree_method": "gpu_hist", "predictor": "gpu_predictor"}


def suggest_xgb_params(trial: optuna.Trial) -> dict:
    return {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }


def build_xgb_estimator(params: dict, y_train: np.ndarray, random_state: int, use_gpu: bool):
    return xgboost.XGBClassifier(
        **params,
        **xgb_gpu_params(use_gpu),
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=positive_weight(y_train),
        random_state=random_state,
        n_jobs=-1,
    )


## LightGBM baseline


In [ ]:
def lgbm_gpu_params(use_gpu: bool) -> dict:
    return {"device_type": "gpu"} if use_gpu else {"device_type": "cpu"}


def suggest_lgbm_params(trial: optuna.Trial) -> dict:
    return {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_categorical("max_depth", [-1, 3, 5, 7, 9, 12]),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }


def build_lgbm_estimator(params: dict, y_train: np.ndarray, random_state: int, use_gpu: bool):
    return lightgbm.LGBMClassifier(
        **params,
        **lgbm_gpu_params(use_gpu),
        objective="binary",
        scale_pos_weight=positive_weight(y_train),
        random_state=random_state,
        n_jobs=-1,
        verbosity=-1,
    )


## ML baseline dispatch 함수


In [ ]:
def suggest_ml_params(trial: optuna.Trial, model_name: str) -> dict:
    if model_name == "LR":
        return suggest_lr_params(trial)
    if model_name == "RF":
        return suggest_rf_params(trial)
    if model_name == "XGB":
        return suggest_xgb_params(trial)
    if model_name == "LGBM":
        return suggest_lgbm_params(trial)
    raise ValueError(f"Unknown ML model: {model_name}")


def build_ml_estimator(model_name: str, params: dict, y_train: np.ndarray, random_state: int, use_gpu: bool):
    if model_name == "LR":
        return build_lr_estimator(params, random_state)
    if model_name == "RF":
        return build_rf_estimator(params, random_state)
    if model_name == "XGB":
        return build_xgb_estimator(params, y_train, random_state, use_gpu)
    if model_name == "LGBM":
        return build_lgbm_estimator(params, y_train, random_state, use_gpu)
    raise ValueError(f"Unknown ML model: {model_name}")


## ML baseline tuning 함수


In [ ]:
def run_ml_tuning(
    model_name: str,
    folds: list[dict],
    output_dir: Path,
    model_dir: Path,
    n_trials: int,
    random_state: int,
    use_gpu: bool,
) -> dict:
    trial_rows = []
    fold_rows = []
    use_scaler = model_name == "LR"

    def objective(trial: optuna.Trial) -> float:
        params = suggest_ml_params(trial, model_name)
        current_fold_rows = []
        for fold in folds:
            train_mask = fold["train_mask"]
            val_mask = fold["val_mask"]
            estimator = build_ml_estimator(model_name, params, y_train_within_t_plus_2[train_mask], random_state, use_gpu)
            estimator, scaler, fallback_reason = fit_ml_estimator(
                model_name,
                estimator,
                X_train_t[train_mask],
                y_train_within_t_plus_2[train_mask],
                use_scaler=use_scaler,
            )
            val_prob = predict_ml_proba(estimator, scaler, X_train_t[val_mask])
            metrics = binary_metric_dict(y_train_within_t_plus_2[val_mask], val_prob)
            row = {
                "model": model_name,
                "trial_id": trial.number + 1,
                "optuna_trial_number": trial.number,
                "fold_id": fold["fold_id"],
                "gpu_requested": bool(use_gpu and model_name in {"XGB", "LGBM"}),
                "fallback_reason": fallback_reason,
                **params,
                **metrics,
            }
            current_fold_rows.append(row)
            fold_rows.append(row)

        fold_df = pd.DataFrame(current_fold_rows)
        metric_cols = ["auroc", "auprc", "sensitivity", "specificity", "ppv", "npv"]
        row = {
            "model": model_name,
            "trial_id": trial.number + 1,
            "optuna_trial_number": trial.number,
            **params,
            **fold_df[metric_cols].mean(numeric_only=True).add_prefix("cv_mean_").to_dict(),
            **fold_df[metric_cols].std(numeric_only=True).add_prefix("cv_std_").to_dict(),
        }
        trial_rows.append(row)
        trial.set_user_attr("metrics", row)
        return row["cv_mean_auprc"]

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=random_state),
        study_name=f"{model_name.lower()}_t_point_within_t_plus_2",
    )
    study.optimize(objective, n_trials=n_trials)

    tuning_results = pd.DataFrame(trial_rows).sort_values(["cv_mean_auprc", "cv_mean_auroc"], ascending=False)
    cv_fold_metrics = pd.DataFrame(fold_rows)
    tuning_results.to_csv(output_dir / f"{model_name.lower()}_t_point_tuning_results.csv", index=False)
    cv_fold_metrics.to_csv(output_dir / f"{model_name.lower()}_t_point_cv_fold_metrics.csv", index=False)
    study.trials_dataframe(attrs=("number", "value", "params", "state")).to_csv(
        output_dir / f"{model_name.lower()}_t_point_optuna_trials.csv",
        index=False,
    )

    best_params = study.best_trial.params
    estimator = build_ml_estimator(model_name, best_params, y_train_within_t_plus_2, random_state, use_gpu)
    estimator, scaler, fallback_reason = fit_ml_estimator(
        model_name,
        estimator,
        X_train_t,
        y_train_within_t_plus_2,
        use_scaler=use_scaler,
    )
    test_prob = predict_ml_proba(estimator, scaler, X_test_t)
    test_metrics = binary_metric_dict(y_test_within_t_plus_2, test_prob)
    test_metrics.update(
        {
            "model": model_name,
            "split": "test",
            "target": "within_t_plus_2",
            "input": "current_t_point_only",
            "optuna_best_value": float(study.best_value),
            "gpu_requested": bool(use_gpu and model_name in {"XGB", "LGBM"}),
            "fallback_reason": fallback_reason,
            **best_params,
        }
    )

    predictions = meta_test.copy()
    predictions["y_within_t_plus_2_true"] = y_test_within_t_plus_2.astype(int)
    predictions[f"{model_name.lower()}_t_point_prob"] = test_prob
    predictions[f"{model_name.lower()}_t_point_pred_0_5"] = (test_prob >= 0.5).astype(int)
    pd.DataFrame([test_metrics]).to_csv(output_dir / f"{model_name.lower()}_t_point_test_metrics.csv", index=False)
    predictions.to_csv(output_dir / f"{model_name.lower()}_t_point_test_predictions.csv", index=False)
    joblib.dump(
        {
            "model": estimator,
            "scaler": scaler,
            "best_params": best_params,
            "cv_metrics": study.best_trial.user_attrs["metrics"],
            "test_metrics": test_metrics,
            "input": "current_t_point_only",
            "target": "within_t_plus_2",
        },
        model_dir / f"{model_name.lower()}_t_point_within_t_plus_2.joblib",
    )
    return test_metrics


## MLP baseline 모델과 DataLoader


In [ ]:
class MLPTPointClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, dropout: float):
        super().__init__()
        layers = []
        in_size = input_size
        for _ in range(num_layers):
            layers.extend(
                [
                    nn.Linear(in_size, hidden_size),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ]
            )
            in_size = hidden_size
        layers.append(nn.Linear(in_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)


def make_binary_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
    pin_memory: bool,
    random_state: int,
) -> DataLoader:
    dataset = TensorDataset(torch.from_numpy(x).float(), torch.from_numpy(y).float())
    generator = torch.Generator().manual_seed(random_state)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
        pin_memory=pin_memory,
    )


def predict_binary_torch_proba(
    model: nn.Module,
    x: np.ndarray,
    batch_size: int,
    device: torch.device,
    non_blocking: bool,
    amp_enabled: bool,
) -> np.ndarray:
    model.eval()
    dummy_y = np.zeros(len(x), dtype=np.float32)
    loader = make_binary_loader(x, dummy_y, batch_size, shuffle=False, pin_memory=device.type == "cuda", random_state=0)
    probs = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device, non_blocking=non_blocking)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(xb)
            probs.append(torch.sigmoid(logits.float()).detach().cpu().numpy())
    return np.concatenate(probs, axis=0)


## MLP baseline 탐색 공간


In [ ]:
def suggest_mlp_params(trial: optuna.Trial) -> dict:
    return {
        "hidden_size": trial.suggest_categorical("hidden_size", [64, 128, 256, 512]),
        "num_layers": trial.suggest_int("num_layers", 1, 3),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
    }


## MLP baseline fold 학습 함수


In [ ]:
def train_mlp_fold(
    params: dict,
    fold: dict,
    max_epochs: int,
    patience: int,
    random_state: int,
    device: torch.device,
    amp_enabled: bool,
) -> tuple[dict, dict]:
    train_mask = fold["train_mask"]
    val_mask = fold["val_mask"]
    pin_memory = device.type == "cuda"
    non_blocking = device.type == "cuda"

    scaler_obj = StandardScaler()
    x_fold_train = scaler_obj.fit_transform(X_train_t[train_mask]).astype(np.float32)
    x_fold_val = scaler_obj.transform(X_train_t[val_mask]).astype(np.float32)

    model = MLPTPointClassifier(
        input_size=X_train_t.shape[1],
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
    ).to(device)
    pos_weight = torch.tensor([positive_weight(y_train_within_t_plus_2[train_mask])], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    train_loader = make_binary_loader(
        x_fold_train,
        y_train_within_t_plus_2[train_mask],
        params["batch_size"],
        shuffle=True,
        pin_memory=pin_memory,
        random_state=random_state,
    )

    best_state = None
    best_score = -np.inf
    best_epoch = 0
    epochs_without_improve = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=non_blocking)
            yb = yb.to(device, non_blocking=non_blocking)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        val_prob = predict_binary_torch_proba(model, x_fold_val, params["batch_size"], device, non_blocking, amp_enabled)
        metrics = binary_metric_dict(y_train_within_t_plus_2[val_mask], val_prob)
        score = metrics["auprc"] if not np.isnan(metrics["auprc"]) else -np.inf
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), **metrics})

        if best_state is None or score > best_score:
            best_score = score
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= patience:
                break

    model.load_state_dict(best_state)
    val_prob = predict_binary_torch_proba(model, x_fold_val, params["batch_size"], device, non_blocking, amp_enabled)
    final_metrics = binary_metric_dict(y_train_within_t_plus_2[val_mask], val_prob)
    final_metrics.update({"fold_id": fold["fold_id"], "best_epoch": best_epoch, "epochs_run": len(history), "best_score": best_score})

    if device.type == "cuda":
        torch.cuda.empty_cache()
    return final_metrics, {"history": history, "state_dict": best_state, "scaler": scaler_obj}


## MLP baseline 전체 train 재학습 함수


In [ ]:
def train_mlp_full(
    params: dict,
    epochs: int,
    random_state: int,
    device: torch.device,
    amp_enabled: bool,
) -> dict:
    pin_memory = device.type == "cuda"
    non_blocking = device.type == "cuda"
    scaler_obj = StandardScaler()
    x_train_scaled = scaler_obj.fit_transform(X_train_t).astype(np.float32)

    model = MLPTPointClassifier(
        input_size=X_train_t.shape[1],
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
    ).to(device)
    pos_weight = torch.tensor([positive_weight(y_train_within_t_plus_2)], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    train_loader = make_binary_loader(x_train_scaled, y_train_within_t_plus_2, params["batch_size"], True, pin_memory, random_state)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=non_blocking)
            yb = yb.to(device, non_blocking=non_blocking)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses))})

    state_dict = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return {"state_dict": state_dict, "history": history, "epochs_run": epochs, "scaler": scaler_obj}


## MLP baseline tuning 함수


In [ ]:
def run_mlp_tuning(
    folds: list[dict],
    output_dir: Path,
    model_dir: Path,
    n_trials: int,
    max_epochs: int,
    patience: int,
    random_state: int,
    device: torch.device,
    amp_enabled: bool,
) -> dict:
    trial_rows = []
    fold_rows = []
    history_rows = []

    def objective(trial: optuna.Trial) -> float:
        params = suggest_mlp_params(trial)
        current_rows = []
        fold_epochs = []
        for fold in folds:
            metrics, artifact = train_mlp_fold(params, fold, max_epochs, patience, random_state, device, amp_enabled)
            row = {
                "model": "MLP",
                "trial_id": trial.number + 1,
                "optuna_trial_number": trial.number,
                **params,
                **metrics,
            }
            current_rows.append(row)
            fold_rows.append(row)
            fold_epochs.append(metrics["best_epoch"])

            history_df = pd.DataFrame(artifact["history"])
            history_df.insert(0, "fold_id", fold["fold_id"])
            history_df.insert(0, "trial_id", trial.number + 1)
            history_df.insert(0, "optuna_trial_number", trial.number)
            history_rows.append(history_df)

        fold_df = pd.DataFrame(current_rows)
        metric_cols = ["auroc", "auprc", "sensitivity", "specificity", "ppv", "npv", "best_epoch", "epochs_run"]
        row = {
            "model": "MLP",
            "trial_id": trial.number + 1,
            "optuna_trial_number": trial.number,
            **params,
            "final_epochs": int(max(1, round(float(np.mean(fold_epochs))))),
            **fold_df[metric_cols].mean(numeric_only=True).add_prefix("cv_mean_").to_dict(),
            **fold_df[metric_cols].std(numeric_only=True).add_prefix("cv_std_").to_dict(),
        }
        trial_rows.append(row)
        trial.set_user_attr("metrics", row)
        trial.set_user_attr("final_epochs", row["final_epochs"])
        return row["cv_mean_auprc"]

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=random_state),
        study_name="mlp_t_point_within_t_plus_2",
    )
    study.optimize(objective, n_trials=n_trials)

    tuning_results = pd.DataFrame(trial_rows).sort_values(["cv_mean_auprc", "cv_mean_auroc"], ascending=False)
    pd.DataFrame(fold_rows).to_csv(output_dir / "mlp_t_point_cv_fold_metrics.csv", index=False)
    tuning_results.to_csv(output_dir / "mlp_t_point_tuning_results.csv", index=False)
    if history_rows:
        pd.concat(history_rows, ignore_index=True).to_csv(output_dir / "mlp_t_point_cv_fold_history.csv", index=False)
    study.trials_dataframe(attrs=("number", "value", "params", "state")).to_csv(
        output_dir / "mlp_t_point_optuna_trials.csv",
        index=False,
    )

    best_params = study.best_trial.params
    final_artifact = train_mlp_full(
        best_params,
        epochs=study.best_trial.user_attrs["final_epochs"],
        random_state=random_state,
        device=device,
        amp_enabled=amp_enabled,
    )
    model = MLPTPointClassifier(
        input_size=X_train_t.shape[1],
        hidden_size=best_params["hidden_size"],
        num_layers=best_params["num_layers"],
        dropout=best_params["dropout"],
    ).to(device)
    model.load_state_dict(final_artifact["state_dict"])
    x_test_scaled = final_artifact["scaler"].transform(X_test_t).astype(np.float32)
    test_prob = predict_binary_torch_proba(model, x_test_scaled, best_params["batch_size"], device, device.type == "cuda", amp_enabled)
    test_metrics = binary_metric_dict(y_test_within_t_plus_2, test_prob)
    test_metrics.update(
        {
            "model": "MLP",
            "split": "test",
            "target": "within_t_plus_2",
            "input": "current_t_point_only",
            "optuna_best_value": float(study.best_value),
            "optuna_best_trial_number": int(study.best_trial.number),
            "final_epochs": final_artifact["epochs_run"],
            "device": str(device),
            **best_params,
        }
    )

    predictions = meta_test.copy()
    predictions["y_within_t_plus_2_true"] = y_test_within_t_plus_2.astype(int)
    predictions["mlp_t_point_prob"] = test_prob
    predictions["mlp_t_point_pred_0_5"] = (test_prob >= 0.5).astype(int)
    pd.DataFrame([test_metrics]).to_csv(output_dir / "mlp_t_point_test_metrics.csv", index=False)
    predictions.to_csv(output_dir / "mlp_t_point_test_predictions.csv", index=False)

    torch.save(
        {
            "model_architecture": "mlp_t_point_within_t_plus_2",
            "model_state_dict": final_artifact["state_dict"],
            "scaler": final_artifact["scaler"],
            "params": best_params,
            "cv_metrics": study.best_trial.user_attrs["metrics"],
            "optuna_best_value": float(study.best_value),
            "optuna_best_trial_number": int(study.best_trial.number),
            "final_train_history": final_artifact["history"],
            "test_metrics": test_metrics,
            "input_size": int(X_train_t.shape[1]),
            "output_size": 1,
            "target": "within_t_plus_2",
            "device": str(device),
            "amp_enabled": amp_enabled,
        },
        model_dir / "mlp_t_point_within_t_plus_2_best_model.pt",
    )
    return test_metrics


## Single-output LSTM 모델과 DataLoader


In [ ]:
class LSTMWithinClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, dropout: float):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, 1))

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        return self.head(hidden[-1]).squeeze(-1)


def make_lstm_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
    pin_memory: bool,
    random_state: int,
) -> DataLoader:
    dataset = TensorDataset(torch.from_numpy(x).float(), torch.from_numpy(y).float())
    generator = torch.Generator().manual_seed(random_state)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
        pin_memory=pin_memory,
    )


def predict_lstm_proba(model: nn.Module, x: np.ndarray, batch_size: int, device: torch.device, non_blocking: bool, amp_enabled: bool) -> np.ndarray:
    model.eval()
    dummy_y = np.zeros(len(x), dtype=np.float32)
    loader = make_lstm_loader(x, dummy_y, batch_size, shuffle=False, pin_memory=device.type == "cuda", random_state=0)
    probs = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device, non_blocking=non_blocking)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(xb)
            probs.append(torch.sigmoid(logits.float()).detach().cpu().numpy())
    return np.concatenate(probs, axis=0)


## Single-output LSTM 탐색 공간


In [ ]:
def suggest_lstm_params(trial: optuna.Trial) -> dict:
    num_layers = trial.suggest_int("num_layers", 1, 3)
    return {
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128, 256]),
        "num_layers": num_layers,
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
    }


## Single-output LSTM fold 학습 함수


In [ ]:
def train_lstm_fold(
    params: dict,
    fold: dict,
    max_epochs: int,
    patience: int,
    random_state: int,
    device: torch.device,
    amp_enabled: bool,
) -> tuple[dict, dict]:
    train_mask = fold["train_mask"]
    val_mask = fold["val_mask"]
    pin_memory = device.type == "cuda"
    non_blocking = device.type == "cuda"

    model = LSTMWithinClassifier(
        input_size=X_train.shape[2],
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
    ).to(device)
    pos_weight = torch.tensor([positive_weight(y_train_within_t_plus_2[train_mask])], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    train_loader = make_lstm_loader(
        X_train[train_mask],
        y_train_within_t_plus_2[train_mask],
        params["batch_size"],
        shuffle=True,
        pin_memory=pin_memory,
        random_state=random_state,
    )

    best_state = None
    best_score = -np.inf
    best_epoch = 0
    epochs_without_improve = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=non_blocking)
            yb = yb.to(device, non_blocking=non_blocking)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        val_prob = predict_lstm_proba(
            model,
            X_train[val_mask],
            params["batch_size"],
            device,
            non_blocking,
            amp_enabled,
        )
        metrics = binary_metric_dict(y_train_within_t_plus_2[val_mask], val_prob)
        score = metrics["auprc"] if not np.isnan(metrics["auprc"]) else -np.inf
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), **metrics})

        if best_state is None or score > best_score:
            best_score = score
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= patience:
                break

    model.load_state_dict(best_state)
    val_prob = predict_lstm_proba(model, X_train[val_mask], params["batch_size"], device, non_blocking, amp_enabled)
    final_metrics = binary_metric_dict(y_train_within_t_plus_2[val_mask], val_prob)
    final_metrics.update({"fold_id": fold["fold_id"], "best_epoch": best_epoch, "epochs_run": len(history), "best_score": best_score})

    if device.type == "cuda":
        torch.cuda.empty_cache()
    return final_metrics, {"history": history, "state_dict": best_state}


## Single-output LSTM 전체 train 재학습 함수


In [ ]:
def train_lstm_full(
    params: dict,
    epochs: int,
    random_state: int,
    device: torch.device,
    amp_enabled: bool,
) -> dict:
    pin_memory = device.type == "cuda"
    non_blocking = device.type == "cuda"
    model = LSTMWithinClassifier(
        input_size=X_train.shape[2],
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
    ).to(device)
    pos_weight = torch.tensor([positive_weight(y_train_within_t_plus_2)], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    train_loader = make_lstm_loader(X_train, y_train_within_t_plus_2, params["batch_size"], True, pin_memory, random_state)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=non_blocking)
            yb = yb.to(device, non_blocking=non_blocking)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses))})

    state_dict = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return {"state_dict": state_dict, "history": history, "epochs_run": epochs}


## Single-output LSTM tuning 함수


In [ ]:
def run_lstm_tuning(
    folds: list[dict],
    output_dir: Path,
    model_dir: Path,
    n_trials: int,
    max_epochs: int,
    patience: int,
    random_state: int,
    device: torch.device,
    amp_enabled: bool,
) -> dict:
    trial_rows = []
    fold_rows = []
    history_rows = []

    def objective(trial: optuna.Trial) -> float:
        params = suggest_lstm_params(trial)
        current_rows = []
        fold_epochs = []
        for fold in folds:
            metrics, artifact = train_lstm_fold(params, fold, max_epochs, patience, random_state, device, amp_enabled)
            row = {
                "model": "LSTM",
                "trial_id": trial.number + 1,
                "optuna_trial_number": trial.number,
                **params,
                **metrics,
            }
            current_rows.append(row)
            fold_rows.append(row)
            fold_epochs.append(metrics["best_epoch"])

            history_df = pd.DataFrame(artifact["history"])
            history_df.insert(0, "fold_id", fold["fold_id"])
            history_df.insert(0, "trial_id", trial.number + 1)
            history_df.insert(0, "optuna_trial_number", trial.number)
            history_rows.append(history_df)

        fold_df = pd.DataFrame(current_rows)
        metric_cols = ["auroc", "auprc", "sensitivity", "specificity", "ppv", "npv", "best_epoch", "epochs_run"]
        row = {
            "model": "LSTM",
            "trial_id": trial.number + 1,
            "optuna_trial_number": trial.number,
            **params,
            "final_epochs": int(max(1, round(float(np.mean(fold_epochs))))),
            **fold_df[metric_cols].mean(numeric_only=True).add_prefix("cv_mean_").to_dict(),
            **fold_df[metric_cols].std(numeric_only=True).add_prefix("cv_std_").to_dict(),
        }
        trial_rows.append(row)
        trial.set_user_attr("metrics", row)
        trial.set_user_attr("final_epochs", row["final_epochs"])
        return row["cv_mean_auprc"]

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=random_state),
        study_name="single_output_lstm_within_t_plus_2",
    )
    study.optimize(objective, n_trials=n_trials)

    tuning_results = pd.DataFrame(trial_rows).sort_values(["cv_mean_auprc", "cv_mean_auroc"], ascending=False)
    pd.DataFrame(fold_rows).to_csv(output_dir / "lstm_within_t_plus_2_cv_fold_metrics.csv", index=False)
    tuning_results.to_csv(output_dir / "lstm_within_t_plus_2_tuning_results.csv", index=False)
    if history_rows:
        pd.concat(history_rows, ignore_index=True).to_csv(output_dir / "lstm_within_t_plus_2_cv_fold_history.csv", index=False)
    study.trials_dataframe(attrs=("number", "value", "params", "state")).to_csv(
        output_dir / "lstm_within_t_plus_2_optuna_trials.csv",
        index=False,
    )

    best_params = study.best_trial.params
    final_artifact = train_lstm_full(
        best_params,
        epochs=study.best_trial.user_attrs["final_epochs"],
        random_state=random_state,
        device=device,
        amp_enabled=amp_enabled,
    )
    model = LSTMWithinClassifier(
        input_size=X_train.shape[2],
        hidden_size=best_params["hidden_size"],
        num_layers=best_params["num_layers"],
        dropout=best_params["dropout"],
    ).to(device)
    model.load_state_dict(final_artifact["state_dict"])
    test_prob = predict_lstm_proba(model, X_test, best_params["batch_size"], device, device.type == "cuda", amp_enabled)
    test_metrics = binary_metric_dict(y_test_within_t_plus_2, test_prob)
    test_metrics.update(
        {
            "model": "LSTM",
            "split": "test",
            "target": "within_t_plus_2",
            "input": "sequence_t_minus_3_to_t",
            "optuna_best_value": float(study.best_value),
            "optuna_best_trial_number": int(study.best_trial.number),
            "final_epochs": final_artifact["epochs_run"],
            **best_params,
        }
    )

    predictions = meta_test.copy()
    predictions["y_within_t_plus_2_true"] = y_test_within_t_plus_2.astype(int)
    predictions["lstm_within_t_plus_2_prob"] = test_prob
    predictions["lstm_within_t_plus_2_pred_0_5"] = (test_prob >= 0.5).astype(int)
    pd.DataFrame([test_metrics]).to_csv(output_dir / "lstm_within_t_plus_2_test_metrics.csv", index=False)
    predictions.to_csv(output_dir / "lstm_within_t_plus_2_test_predictions.csv", index=False)

    torch.save(
        {
            "model_architecture": "lstm_single_output_within_t_plus_2",
            "model_state_dict": final_artifact["state_dict"],
            "params": best_params,
            "cv_metrics": study.best_trial.user_attrs["metrics"],
            "optuna_best_value": float(study.best_value),
            "optuna_best_trial_number": int(study.best_trial.number),
            "final_train_history": final_artifact["history"],
            "test_metrics": test_metrics,
            "input_size": int(X_train.shape[2]),
            "sequence_length": int(X_train.shape[1]),
            "output_size": 1,
            "target": "within_t_plus_2",
            "device": str(device),
            "amp_enabled": amp_enabled,
        },
        model_dir / "lstm_within_t_plus_2_best_model.pt",
    )
    with open(model_dir / "lstm_within_t_plus_2_best_model_config.json", "w", encoding="utf-8") as f:
        json.dump(
            to_builtin(
                {
                    "model_architecture": "lstm_single_output_within_t_plus_2",
                    "params": best_params,
                    "cv_metrics": study.best_trial.user_attrs["metrics"],
                    "optuna_best_value": float(study.best_value),
                    "optuna_best_trial_number": int(study.best_trial.number),
                    "final_train_history": final_artifact["history"],
                    "test_metrics": test_metrics,
                    "input_size": int(X_train.shape[2]),
                    "sequence_length": int(X_train.shape[1]),
                    "output_size": 1,
                    "target": "within_t_plus_2",
                    "device": str(device),
                    "amp_enabled": amp_enabled,
                }
            ),
            f,
            indent=2,
            ensure_ascii=False,
        )
    return test_metrics


## ML baseline hyperparameter tuning

LR/RF/XGB/LightGBM은 모두 anchor `t` 시점 feature만 사용합니다. XGB와 LightGBM은 CUDA가 사용 가능하면 GPU parameter를 우선 적용합니다.


In [ ]:
package_versions = pd.DataFrame(
    [
        {"package": "xgboost", "version": xgboost.__version__},
        {"package": "lightgbm", "version": lightgbm.__version__},
        {"package": "torch", "version": torch.__version__},
    ]
)
display(package_versions)


In [ ]:
ml_test_metric_rows = []
for model_name in ML_MODEL_NAMES:
    print()
    print(f"=== {model_name}: current t-point ML baseline ===")
    ml_test_metric_rows.append(
        run_ml_tuning(
            model_name=model_name,
            folds=cv_folds,
            output_dir=OUTPUT_DIR,
            model_dir=MODEL_DIR,
            n_trials=N_TRIALS_ML,
            random_state=RANDOM_STATE,
            use_gpu=device.type == "cuda",
        )
    )

ml_test_metrics = pd.DataFrame(ml_test_metric_rows)
if not ml_test_metrics.empty:
    display(ml_test_metrics.sort_values("auprc", ascending=False))


## MLP baseline hyperparameter tuning

MLP는 LR/RF/XGB/LightGBM과 동일하게 anchor `t` 시점 feature만 사용하는 PyTorch deep learning baseline입니다. CUDA가 있으면 GPU/AMP를 사용합니다.


In [ ]:
if RUN_MLP:
    print("=== MLP: current t-point deep learning baseline ===")
    mlp_test_metrics = run_mlp_tuning(
        folds=cv_folds,
        output_dir=OUTPUT_DIR,
        model_dir=MODEL_DIR,
        n_trials=N_TRIALS_MLP,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        random_state=RANDOM_STATE,
        device=device,
        amp_enabled=device.type == "cuda",
    )
    display(pd.DataFrame([mlp_test_metrics]))
else:
    mlp_test_metrics = None


## Single-output LSTM hyperparameter tuning

기존 encoder-decoder multi-horizon 구조 대신, `t-3~t` input sequence를 LSTM이 읽고 마지막 hidden state에서 `within_t_plus_2` logit 하나를 출력합니다.


In [ ]:
if RUN_LSTM:
    lstm_test_metrics = run_lstm_tuning(
        folds=cv_folds,
        output_dir=OUTPUT_DIR,
        model_dir=MODEL_DIR,
        n_trials=N_TRIALS_LSTM,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        random_state=RANDOM_STATE,
        device=device,
        amp_enabled=device.type == "cuda",
    )
    display(pd.DataFrame([lstm_test_metrics]))
else:
    lstm_test_metrics = None


## 최종 결과 요약 저장


In [ ]:
summary_rows = []
if not ml_test_metrics.empty:
    summary_rows.extend(ml_test_metrics.to_dict(orient="records"))
if mlp_test_metrics is not None:
    summary_rows.append(mlp_test_metrics)
if lstm_test_metrics is not None:
    summary_rows.append(lstm_test_metrics)

within_t_plus_2_test_summary = pd.DataFrame(summary_rows)
if not within_t_plus_2_test_summary.empty:
    within_t_plus_2_test_summary = within_t_plus_2_test_summary.sort_values("auprc", ascending=False).reset_index(drop=True)
    within_t_plus_2_test_summary.to_csv(OUTPUT_DIR / "within_t_plus_2_test_metrics_summary.csv", index=False)
    display(within_t_plus_2_test_summary)

print("Saved within t+2 ML/MLP baselines and single-output LSTM outputs")
